<a href="https://colab.research.google.com/github/vaibhavm291/PROTOTYPES-/blob/main/Trial_Terrain_intelligence_engine_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
import logging
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.mask import mask
from rasterio.warp import transform_geom
from shapely.geometry import shape, mapping, box
from ultralytics import YOLO

# Professional Logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class TerrainIntelligenceEngine:
    def __init__(self, dem_path, model_path=None):
        self.dem_path = dem_path
        self.model = YOLO(model_path) if model_path else None

        with rasterio.open(dem_path) as src:
            self.dem_crs = src.crs
            self.dem_transform = src.transform
            self.dem_nodata = src.nodata
            self.dem_res = src.res[0] # Pixel resolution (e.g., 30m)

    def spatial_handshake(self, image_path, detections):
        """
        Translates Pixel (x, y) detections to Geographic Coordinates
        and reprojects them to the DEM's CRS.
        """
        logging.info("Initiating Spatial Handshake...")
        with rasterio.open(image_path) as img_src:
            img_crs = img_src.crs
            img_transform = img_src.transform

        georeferenced_polygons = []
        for box_data in detections.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box_data

            # Pixel to Image Coordinates
            lon1, lat1 = rasterio.transform.xy(img_transform, y1, x1)
            lon2, lat2 = rasterio.transform.xy(img_transform, y2, x2)

            # Create geometry and reproject to DEM CRS
            poly = box(lon1, lat2, lon2, lat1)
            geom_reprojected = transform_geom(img_crs, self.dem_crs, mapping(poly))
            georeferenced_polygons.append(shape(geom_reprojected))

        return gpd.GeoDataFrame(geometry=georeferenced_polygons, crs=self.dem_crs)

    def z_axis_validation(self, gdf, height_threshold=1.5):
        """
        Validates CV detections using Z-Axis (Height) data from the DEM.
        This filters out false positives like 'blue tarps' being flagged as 'ponds'.
        """
        valid_results = []
        with rasterio.open(self.dem_path) as src:
            for geom in gdf.geometry:
                # Clip DEM to the detected object's mask
                out_image, _ = mask(src, [mapping(geom)], crop=True)
                data = out_image[0]
                valid_data = data[data != self.dem_nodata]

                if valid_data.size == 0: continue

                z_min, z_max = np.min(valid_data), np.max(valid_data)
                height = z_max - z_min

                # Logic Refinement: Verify if the object height matches the classification
                # Example: A 'Car' shouldn't be 10 meters tall.
                classification = "Elevated Structure" if height > height_threshold else "Ground Asset"

                valid_results.append({
                    "geometry": geom,
                    "height": float(height),
                    "mean_elev": float(np.mean(valid_data)),
                    "class_validation": classification,
                    "volume_m3": float(len(valid_data) * (self.dem_res**2) * height)
                })

        return gpd.GeoDataFrame(valid_results, crs=self.dem_crs)

    def run_hydrology_impact(self, gdf, flow_acc_path):
        """
        Predicting the 'Unseen': Overlays assets onto Flow Accumulation paths.
        """
        # Load flow accumulation from your hydrology engine
        with rasterio.open(flow_acc_path) as src:
            # Spatial join logic to see which assets intersect high-flow paths
            pass
        return gdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import logging
import rasterio
import numpy as np
import geopandas as gpd
import os
from google.colab import files
from rasterio.mask import mask
from rasterio.warp import transform_geom
from shapely.geometry import shape, mapping, box
from ultralytics import YOLO
from rasterio import features

# Set up industrial logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class TerrainIntelligenceEngine:
    def __init__(self, dem_path, model_path=None):
        self.dem_path = dem_path
        self.model = YOLO(model_path) if model_path else None

        with rasterio.open(dem_path) as src:
            self.dem_crs = src.crs
            self.dem_res = src.res[0]
            self.dem_nodata = src.nodata

    def _spatial_handshake(self, image_path, detections):
        """Aligns Drone Pixels with Global GIS Coordinates."""
        with rasterio.open(image_path) as img_src:
            img_crs = img_src.crs
            img_transform = img_src.transform

        polygons = []
        for box_data in detections.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box_data
            lon1, lat1 = rasterio.transform.xy(img_transform, y1, x1)
            lon2, lat2 = rasterio.transform.xy(img_transform, y2, x2)
            poly = box(lon1, lat2, lon2, lat1)

            geom_reprojected = transform_geom(img_crs, self.dem_crs, mapping(poly))
            polygons.append(shape(geom_reprojected))

        return gpd.GeoDataFrame(geometry=polygons, crs=self.dem_crs)

    def run_mode_1_fusion(self, image_path, civic_infra_path=None):
        """Fusion Mode: CV + DEM + GIS (Urban Analysis)"""
        logging.info("Analyzing CV Detections and Aligning to Terrain...")
        results = self.model(image_path)[0]
        gdf = self._spatial_handshake(image_path, results)

        valid_data = []
        with rasterio.open(self.dem_path) as src:
            for geom in gdf.geometry:
                clipped, _ = mask(src, [mapping(geom)], crop=True)
                vals = clipped[0][clipped[0] != self.dem_nodata]

                if vals.size > 0:
                    height = float(np.max(vals) - np.min(vals))

                    # Industrial Recommendation Logic
                    if height > 5.0:
                        rec = "Structural Reinforcement Required"
                    elif height < 1.0:
                        rec = "Surface Level Asset - Low Risk"
                    else:
                        rec = "Standard Foundation Applicable"

                    valid_data.append({
                        "geometry": geom,
                        "height_m": height,
                        "vol_m3": float(len(vals)*(self.dem_res**2)*height),
                        "recommendation": rec,
                        "risk_score": float((height * 0.7) + (self.dem_res * 0.3))
                    })

        if not valid_data:
            return gpd.GeoDataFrame(columns=['geometry', 'recommendation'], crs=self.dem_crs)

        output_gdf = gpd.GeoDataFrame(valid_data, crs=self.dem_crs)

        if civic_infra_path and os.path.exists(civic_infra_path):
            infra = gpd.read_file(civic_infra_path).to_crs(self.dem_crs)
            output_gdf = gpd.sjoin(output_gdf, infra, how="inner", predicate="intersects")

        return output_gdf

    def run_mode_2_terrain(self, flow_path):
        """Mode 2: Vectorizing Hydrology & Slope Risk for Engineers"""
        logging.info("Vectorizing Hydrology Risk Zones...")

        with rasterio.open(flow_path) as src:
            data = src.read(1)
            # Thresholding: Identifying high-risk flood paths
            # NOTE: If your DEM elevation max is 233, and you check for > 500,
            # this will always be empty. Adjust threshold based on your data.
            data_mask = (data > 500).astype(np.uint8)

            shapes_gen = (
                {'properties': {'risk_level': 'High Flood Risk', 'rec': 'Install Deep Drainage'}, 'geometry': s}
                for i, (s, v) in enumerate(features.shapes(data, mask=data_mask, transform=src.transform))
            )
            shapes_list = list(shapes_gen)

        if not shapes_list:
            logging.warning("No high-risk zones detected with current threshold.")
            return gpd.GeoDataFrame(columns=['geometry', 'risk_level', 'rec'], crs=self.dem_crs)

        risk_gdf = gpd.GeoDataFrame.from_features(shapes_list, crs=self.dem_crs)
        return risk_gdf

# ==========================================================
# DYNAMIC USER INTERFACE
# ==========================================================

print("--- STEP 1: UPLOAD CORE DATA ---")
dem_upload = files.upload()
dem_filename = list(dem_upload.keys())[0]

print("\n--- STEP 2: CHOOSE ANALYSIS MODE ---")
mode = input("Select Mode (1: Fusion, 2: Terrain): ")

final_output = None

if mode == "1":
    img_upload = files.upload()
    img_filename = list(img_upload.keys())[0]
    model_upload = files.upload()
    model_filename = list(model_upload.keys())[0]

    engine = TerrainIntelligenceEngine(dem_filename, model_filename)
    final_output = engine.run_mode_1_fusion(img_filename)
else:
    engine = TerrainIntelligenceEngine(dem_filename)
    # Industry Note: Ensure dem_filename here is actually a Flow Accumulation raster,
    # or lower the threshold in run_mode_2_terrain to match elevation values.
    final_output = engine.run_mode_2_terrain(dem_filename)

# ==========================================================
# FINAL INDUSTRIAL OUTPUT
# ==========================================================
if final_output is not None:
    if not final_output.empty:
        final_output.to_file("Industrial_Report.geojson", driver="GeoJSON")
        final_output.drop(columns='geometry').to_csv("Engineering_Metrics.csv", index=False)
        print("\n✅ FILES GENERATED: Industrial_Report.geojson & Engineering_Metrics.csv")
    else:
        print("\n⚠ Analysis successful, but no features met the risk criteria.")

--- STEP 1: UPLOAD CORE DATA ---


Saving P5_PAN_CD_N22_000_E088_000_DEM_30m.tif to P5_PAN_CD_N22_000_E088_000_DEM_30m (2).tif

--- STEP 2: CHOOSE ANALYSIS MODE ---
Select Mode (1: Fusion, 2: Terrain): 2



⚠ Analysis successful, but no features met the risk criteria.


In [ ]:
import logging
import rasterio
import numpy as np
import geopandas as gpd
from google.colab import files
from rasterio.mask import mask
from rasterio.warp import transform_geom
from shapely.geometry import shape, mapping, box
from ultralytics import YOLO
from rasterio import features

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class TerrainIntelligenceEngine:
    def __init__(self, dem_path, model_path=None):
        self.dem_path = dem_path
        self.model = YOLO(model_path) if model_path else None

        with rasterio.open(dem_path) as src:
            self.dem_crs = src.crs
            self.dem_res = src.res[0]
            self.dem_nodata = src.nodata
            self.dem_bounds = src.bounds
            self.dem_data = src.read(1)

    def generate_contours(self, interval=1.0):
        """Standard Output for Architects: Elevation Contour Lines."""
        logging.info(f"Generating {interval}m Contours...")
        # Simplistic contour logic for vector export
        import matplotlib.pyplot as plt
        with rasterio.open(self.dem_path) as src:
            data = src.read(1)
            levels = np.arange(np.min(data), np.max(data), interval)
            # This is a placeholder for actual vector contour extraction
            return f"Contour lines at {interval}m interval generated."

    def calculate_site_stats(self):
        """Engineering Metadata for Project Budgeting."""
        data = self.dem_data[self.dem_data != self.dem_nodata]
        stats = {
            "Min Elevation": np.min(data),
            "Max Elevation": np.max(data),
            "Mean Elevation": np.mean(data),
            "Site Area (m2)": data.size * (self.dem_res ** 2),
            "Resolution": self.dem_res
        }
        return stats

    def run_mode_1_fusion(self, image_path):
        """Mode 1: Fusion (CV + DEM) with Engineering Recommendations."""
        results = self.model(image_path)[0]

        with rasterio.open(image_path) as img_src:
            img_crs, img_transform = img_src.crs, img_src.transform

        valid_assets = []
        for box_data in results.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box_data
            lon1, lat1 = rasterio.transform.xy(img_transform, y1, x1)
            lon2, lat2 = rasterio.transform.xy(img_transform, y2, x2)
            poly = shape(transform_geom(img_crs, self.dem_crs, mapping(box(lon1, lat2, lon2, lat1))))

            # Clip DEM to asset
            with rasterio.open(self.dem_path) as src:
                out_img, _ = mask(src, [mapping(poly)], crop=True)
                z = out_img[0][out_img[0] != self.dem_nodata]

                if z.size > 0:
                    height = np.max(z) - np.min(z)
                    # Engineering Logic
                    rec = "Foundation Check" if height > 3 else "Standard Grade"
                    valid_assets.append({"geometry": poly, "height": height, "recommendation": rec})

        return gpd.GeoDataFrame(valid_assets, crs=self.dem_crs)

    def run_mode_2_terrain(self):
        """Mode 2: Slope & Hydrology (Open Field Survey)."""
        # Calculate Slope (Rise/Run) - Critical for Architects
        x, y = np.gradient(self.dem_data, self.dem_res)
        slope = np.arctan(np.sqrt(x**2 + y**2)) * (180 / np.pi)

        # Vectorize Steep Slopes (>15 degrees is standard restricted building zone)
        mask_steep = (slope > 15).astype(np.uint8)
        shapes = (
            {'properties': {'type': 'Steep Slope', 'rec': 'Retaining Wall Required'}, 'geometry': s}
            for s, v in features.shapes(slope, mask=mask_steep, transform=rasterio.open(self.dem_path).transform)
        )
        return gpd.GeoDataFrame.from_features(list(shapes), crs=self.dem_crs)

# ==========================================================
# EXECUTION & REPORT GENERATION
# ==========================================================
print("Upload DEM...")
dem_file = list(files.upload().keys())[0]
engine = TerrainIntelligenceEngine(dem_file)

mode = input("Select Mode (1: Fusion, 2: Terrain): ")

if mode == "1":
    img_file = list(files.upload().keys())[0]
    model_file = list(files.upload().keys())[0]
    engine.model = YOLO(model_file)
    report_gdf = engine.run_mode_1_fusion(img_file)
else:
    report_gdf = engine.run_mode_2_terrain()

# --- THE PROFESSIONAL HANDOVER ---
stats = engine.calculate_site_stats()
import pandas as pd
pd.DataFrame([stats]).to_csv("SITE_SURVEY_SUMMARY.csv", index=False)
report_gdf.to_file("ENGINEERING_SITE_PLAN.geojson", driver="GeoJSON")

print("\n--- PROFESSIONAL REPORT READY ---")
print("1. ENGINEERING_SITE_PLAN.geojson (For AutoCAD/QGIS)")
print("2. SITE_SURVEY_SUMMARY.csv (For Project Budgeting)")
for k, v in stats.items(): print(f"{k}: {v:.2f}")

Upload DEM...
